# YOLOv12 布匹瑕疵偵測（Colab）

資料集：[Kaggle hcyl1025/guangdong-round1-train2](https://www.kaggle.com/datasets/hcyl1025/guangdong-round1-train2)（天池廣東工業智造 布匹瑕疵 初賽 train2）

流程：下載資料 → 轉成 YOLO 格式 → 檢查 →（選用）Roboflow 標註 → 訓練 → 驗證 → 偵測 → 匯出

**開始前**：選單「執行階段 → 變更執行階段類型」選 **T4 GPU**（或更好的 GPU）。

In [ ]:
!nvidia-smi

## 1. 掛載 Google Drive（訓練結果存在 Drive，斷線也不會不見）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/yolov12_fabric'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

## 2. 下載程式並安裝套件

如果 repo 是 private，請把 URL 改成 `https://<GitHub token>@github.com/star90154/IOT_-.git`（token 放在 Colab Secrets 比較安全）。

In [ ]:
REPO = 'https://github.com/star90154/IOT_-.git'
BRANCH = 'claude/yolov12-setup-annotation-pem62w'   # 合併後改成 main
!rm -rf /content/IOT_- && git clone -b {BRANCH} {REPO} /content/IOT_-
%cd /content/IOT_-
!pip install -q -r requirements.txt
import ultralytics; ultralytics.checks()

## 3. 從 Kaggle 下載資料集

若出現權限錯誤：到 Kaggle → Settings → API → Create New Token，
在 Colab 左側 🔑 Secrets 加入 `KAGGLE_USERNAME`、`KAGGLE_KEY` 兩個值並開啟「Notebook access」。

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception as e:
    print('未設定 Kaggle Secrets，嘗試匿名下載：', e)

import kagglehub
RAW = kagglehub.dataset_download('hcyl1025/guangdong-round1-train2')
print('下載位置：', RAW)

In [ ]:
# 若資料集內是 zip，先解壓到 /content/raw；並列出目錄結構確認
import glob, zipfile, subprocess
zips = glob.glob(f'{RAW}/**/*.zip', recursive=True)
SRC = RAW
if zips:
    SRC = '/content/raw'
    for z in zips:
        print('解壓', z)
        zipfile.ZipFile(z).extractall(SRC)
print(subprocess.run(f'find "{SRC}" -maxdepth 3 -type d | head -30; echo; find "{SRC}" -name "*.json" | head', shell=True, capture_output=True, text=True).stdout)

## 4. 轉成 YOLO 格式

- `--merge20`：套用天池官方 34 → 20 類合併（樣本很少的類別合併後比較好訓練）。想保留原始類別就拿掉。
- `--normal-ratio 0.1`：加入瑕疵圖數量 10% 的無瑕疵圖作為背景，降低誤報。

In [ ]:
DATA = '/content/datasets/fabric'
!python tools/convert_to_yolo.py --src "{SRC}" --out {DATA} --merge20 --normal-ratio 0.1 --val 0.2
!python tools/check_dataset.py {DATA}/data.yaml

### 抽幾張看標註是否正確

In [ ]:
import random, cv2, yaml, matplotlib.pyplot as plt
from pathlib import Path
names = yaml.safe_load(open(f'{DATA}/data.yaml'))['names']
imgs = random.sample(sorted(Path(f'{DATA}/images/train').iterdir()), 4)
fig, axes = plt.subplots(4, 1, figsize=(16, 20))
for ax, p in zip(axes, imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lbl = Path(str(p).replace('/images/', '/labels/')).with_suffix('.txt')
    for line in lbl.read_text().splitlines():
        c, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 4)
        cv2.putText(im, str(int(c)), (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 0, 0), 3)
    ax.imshow(im); ax.set_title(p.name); ax.axis('off')
plt.tight_layout(); print(names)

## 5.（選用）Roboflow：線上檢視 / 修正 / 新增標註

1. 到 [roboflow.com](https://roboflow.com) 建立 workspace，Settings → API Keys 複製 Private API Key
2. Colab Secrets 加 `ROBOFLOW_API_KEY`
3. 執行上傳 → 在 Roboflow 網頁修正標註 → Generate 新版本 → 執行下載，然後把 `DATA` 換成下載的資料夾

沒有要改標註的話可以跳過這一節。

In [ ]:
RF_WORKSPACE = 'your-workspace'   # 改成你的 workspace id
RF_PROJECT = 'fabric-defect'
# os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
# 上傳（資料量大時要一段時間）
# !python tools/roboflow_sync.py upload --data {DATA} --workspace {RF_WORKSPACE} --project {RF_PROJECT}
# 下載第 N 版
# !python tools/roboflow_sync.py download --workspace {RF_WORKSPACE} --project {RF_PROJECT} --version 1 --out /content/datasets/rf
# DATA = '/content/datasets/rf'

## 6. 訓練 YOLOv12

| 模型 | 參數量 | T4 建議 batch (imgsz 1024) |
|---|---|---|
| yolo12n | 2.6M | 16 |
| yolo12s | 9.3M | 8 |
| yolo12m | 20M | 4 |

布匹影像為 2446×1000、瑕疵很小，`imgsz` 建議 ≥ 1024。Colab 斷線後可以用下一格的續訓。

In [ ]:
MODEL = 'yolo12s.pt'
!python train.py --data {DATA}/data.yaml --model {MODEL} --imgsz 1024 --epochs 100 --batch 8 --device 0 \
    --project {SAVE_DIR}/runs --name fabric

In [ ]:
# 斷線續訓（取消註解）
# !python train.py --data {DATA}/data.yaml --model {SAVE_DIR}/runs/fabric/weights/last.pt --resume

## 7. 查看訓練結果

In [ ]:
from IPython.display import Image, display
RUN = sorted(Path(f'{SAVE_DIR}/runs').glob('fabric*'), key=os.path.getmtime)[-1]
print(RUN)
for f in ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    if (RUN / f).exists():
        display(Image(filename=str(RUN / f), width=1000))

## 8. 偵測

In [ ]:
BEST = RUN / 'weights' / 'best.pt'
!python detect.py --weights {BEST} --source {DATA}/images/val --imgsz 1024 --conf 0.25 --json /content/pred.json

In [ ]:
pred_dir = sorted(Path('runs/detect').glob('predict*'), key=os.path.getmtime)[-1]
for p in sorted(pred_dir.glob('*.jpg'))[:4]:
    display(Image(filename=str(p), width=1000))

## 9. 匯出部署用模型（存到 Drive）

In [ ]:
!python export.py --weights {BEST} --format onnx --imgsz 1024
!cp {RUN}/weights/best.onnx {SAVE_DIR}/ && ls -lh {SAVE_DIR}